# 3D Scan Pipeline - Part 2: Training

**Objective:** Train the Gaussian Splatting model using Taichi and Export to .splat.
**Input:** Use the output from Part 1 (`3d_scan_data_part1.zip`) as a **Kaggle Dataset**.

**How to use on Kaggle:**
1. In Part 1, download `3d_scan_data_part1.zip`.
2. Creating a specific Dataset in Kaggle with this file.
3. Add your new Dataset to this notebook (Part 2).

**Environment:** **GPU REQUIRED (T4 or better)**.

In [ ]:
import os
import sys

print("⏳ Setting up Environment (Part 2)...")

# 1. Clone Repo (Only if local project not found)
if not os.path.exists("3DSCAN"):
    !git clone https://github.com/PRIDA-TAKON/3DSCAN.git
    if os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")
else:
    print("📂 Project folder found. Using local version.")
    if os.path.basename(os.getcwd()) != "3DSCAN" and os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")

# 2. Install Dependencies
print("⏳ Installing Dependencies...")
!pip install --upgrade pip
!pip install --upgrade numba scipy pandas scikit-learn opencv-python opencv-python-headless opencv-contrib-python matplotlib pillow plyfile tqdm roma
!pip install taichi

# Install Taichi Splatting (Wanmeihuali Version)
if os.path.exists("taichi_3d_gaussian_splatting"):
    print("📂 taichi_3d_gaussian_splatting found. Skipping clone.")
else:
    !git clone --depth 1 https://github.com/wanmeihuali/taichi_3d_gaussian_splatting.git

# Standard install without forcing numpy version (Using Kaggle Default)
!pip install -r taichi_3d_gaussian_splatting/requirements.txt
!pip install ./taichi_3d_gaussian_splatting

# Verify Environment
try:
    import numpy
    print(f"✅ Setup Complete. NumPy Version: {numpy.__version__}")
except Exception as e:
    print(f"⚠️ Error checking numpy: {e}")

In [ ]:
print("=== FIX UPSTREAM BUGS ===")
# Patch GaussianPointTrainer.py to fix Mutable Default Argument error AND Contiguous Tensor error
import os

target_file = "taichi_3d_gaussian_splatting/taichi_3d_gaussian_splatting/GaussianPointTrainer.py"
if os.path.exists(target_file):
    print(f"🔧 Patching {target_file}...")
    with open(target_file, "r") as f:
        content = f.read()
    
    # 1. Fix for: ValueError: mutable default ... for field rasterisation_config
    if "from dataclasses import dataclass" in content and "from dataclasses import field" not in content:
        content = content.replace("from dataclasses import dataclass", "from dataclasses import dataclass, field")

    replacements = [
        (
            "rasterisation_config: GaussianPointCloudRasterisation.GaussianPointCloudRasterisationConfig = GaussianPointCloudRasterisation.GaussianPointCloudRasterisationConfig()",
            "rasterisation_config: GaussianPointCloudRasterisation.GaussianPointCloudRasterisationConfig = field(default_factory=GaussianPointCloudRasterisation.GaussianPointCloudRasterisationConfig)"
        ),
        (
            "adaptive_controller_config: GaussianPointAdaptiveController.GaussianPointAdaptiveControllerConfig = GaussianPointAdaptiveController.GaussianPointAdaptiveControllerConfig()",
            "adaptive_controller_config: GaussianPointAdaptiveController.GaussianPointAdaptiveControllerConfig = field(default_factory=GaussianPointAdaptiveController.GaussianPointAdaptiveControllerConfig)"
        ),
        (
            "gaussian_point_cloud_scene_config: GaussianPointCloudScene.PointCloudSceneConfig = GaussianPointCloudScene.PointCloudSceneConfig()",
            "gaussian_point_cloud_scene_config: GaussianPointCloudScene.PointCloudSceneConfig = field(default_factory=GaussianPointCloudScene.PointCloudSceneConfig)"
        ),
        (
            "loss_function_config: LossFunction.LossFunctionConfig = LossFunction.LossFunctionConfig()",
            "loss_function_config: LossFunction.LossFunctionConfig = field(default_factory=LossFunction.LossFunctionConfig)"
        )
    ]
    
    for old, new in replacements:
        if old in content:
            content = content.replace(old, new)
    
    # 2. Fix for: ValueError: Non contiguous tensors are not supported
    # We force .contiguous() on inputs to GaussianPointCloudRasterisationInput
    original_block = """
            gaussian_point_cloud_rasterisation_input = GaussianPointCloudRasterisation.GaussianPointCloudRasterisationInput(
                point_cloud=self.scene.point_cloud,
                point_cloud_features=self.scene.point_cloud_features,
                point_object_id=self.scene.point_object_id,
                point_invalid_mask=self.scene.point_invalid_mask,
                camera_info=camera_info,
                q_pointcloud_camera=q_pointcloud_camera,
                t_pointcloud_camera=t_pointcloud_camera,
                color_max_sh_band=iteration // self.config.increase_color_max_sh_band_interval,
            )"""
    
    fixed_block = """
            gaussian_point_cloud_rasterisation_input = GaussianPointCloudRasterisation.GaussianPointCloudRasterisationInput(
                point_cloud=self.scene.point_cloud.contiguous(),
                point_cloud_features=self.scene.point_cloud_features.contiguous(),
                point_object_id=self.scene.point_object_id.contiguous(),
                point_invalid_mask=self.scene.point_invalid_mask.contiguous(),
                camera_info=camera_info,
                q_pointcloud_camera=q_pointcloud_camera.contiguous(),
                t_pointcloud_camera=t_pointcloud_camera.contiguous(),
                color_max_sh_band=iteration // self.config.increase_color_max_sh_band_interval,
            )"""
            
    # Normalize indentation and whitespace for reliable replacement
    if original_block.strip() in content or True: # Force check roughly if exact match fails
         # Try simple string replace first
         if original_block.strip() in content:
             content = content.replace(original_block.strip(), fixed_block.strip())
         else:
             # Fallback: specific line replacements if block doesn't match exactly due to formatting
             content = content.replace("point_cloud=self.scene.point_cloud,", "point_cloud=self.scene.point_cloud.contiguous(),")
             content = content.replace("point_cloud_features=self.scene.point_cloud_features,", "point_cloud_features=self.scene.point_cloud_features.contiguous(),")
             content = content.replace("point_object_id=self.scene.point_object_id,", "point_object_id=self.scene.point_object_id.contiguous(),")
             content = content.replace("point_invalid_mask=self.scene.point_invalid_mask,", "point_invalid_mask=self.scene.point_invalid_mask.contiguous(),")
             content = content.replace("q_pointcloud_camera=q_pointcloud_camera,", "q_pointcloud_camera=q_pointcloud_camera.contiguous(),")
             content = content.replace("t_pointcloud_camera=t_pointcloud_camera,", "t_pointcloud_camera=t_pointcloud_camera.contiguous(),")

    with open(target_file, "w") as f:
        f.write(content)
    print("✅ Patch applied successfully (Dataclass + Contiguous Tensors).")
else:
    print("❌ File not found for patching. Check path.")

In [ ]:
print("=== Import Data from Kaggle Dataset ===")
import glob
import zipfile
import shutil

# Reset working_data to avoid conflicts
if os.path.exists("working_data/3d_scan"):
    shutil.rmtree("working_data/3d_scan", ignore_errors=True)
os.makedirs("working_data/3d_scan", exist_ok=True)

# Look for the dataset zip file in /kaggle/input (standard dataset path)
search_paths = ["/kaggle/input", "input", "."]
zip_path = None

for path in search_paths:
    candidates = glob.glob(f"{path}/**/*.zip", recursive=True)
    for c in candidates:
        if "3d_scan_data_part1" in c or "3d_scan_output" in c:
            zip_path = c
            break
    if zip_path: break

folder_candidates = glob.glob("/kaggle/input/**/sparse/0", recursive=True)
if zip_path:
    print(f"📦 Found data zip: {zip_path}")
    print("⏳ Extracting to working_data...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("✅ Extraction Complete.")
elif folder_candidates:
    print("📂 Found unzipped dataset structure. Copying...")
    # We need the root '3d_scan' folder which contains 'colmap', 'images', 'transforms.json'
    # Path typically looks like: .../working_data/3d_scan/colmap/sparse/0
    # Or: .../3d_scan/sparse/0 (if 'colmap' is implicit)
    
    sparse_0_path = folder_candidates[0]
    # Walk up the tree to find the folder containing 'images' or 'colmap'
    current_path = os.path.dirname(sparse_0_path) # .../sparse
    root_path = None
    
    # Traverse up 4 levels max to identify root
    for _ in range(4):
        parent = os.path.dirname(current_path)
        # Check if 'images' or 'transforms.json' exists here
        if os.path.exists(os.path.join(parent, "images")) or os.path.exists(os.path.join(parent, "transforms.json")):
            root_path = parent
            break
        current_path = parent
    
    if not root_path:
        # Fallback: assume the structure is .../3d_scan/colmap/sparse/0
        # dirname 1: .../colmap/sparse
        # dirname 2: .../colmap
        # dirname 3: .../3d_scan
        colmap_sparse = os.path.dirname(sparse_0_path)
        colmap_dir = os.path.dirname(colmap_sparse)
        root_path = os.path.dirname(colmap_dir)

    print(f"   Detected Source Root: {root_path}")
    print(f"   Copying from {root_path} to working_data/3d_scan...")
    
    # Use shell cp for robustness (handles recursive copy better than python libs sometimes)
    cmd = f'cp -r "{root_path}"/* working_data/3d_scan/'
    res = os.system(cmd)
    
    if res == 0:
        print("✅ Data copied successfully.")
    else:
        print(f"❌ Copy failed with exit code {res}. Trying python fallback...")
        import shutil
        try:
            shutil.copytree(root_path, "working_data/3d_scan", dirs_exist_ok=True)
            print("✅ Data copied successfully (Python fallback).")
        except Exception as e:
             print(f"❌ Python copy failed: {e}")
else:
    print("❌ No data found! Please add the '3d_scan_data_part1' dataset.")
    
# Verify structure
if os.path.exists("working_data/3d_scan/sparse") or os.path.exists("working_data/3d_scan/colmap/sparse"):
    print("✅ Data Validation: Sparse model found.")
else:
    print("⚠️ Warning: 'sparse' folder not found in working_data/3d_scan. Training might fail.")

In [ ]:
print("=== STEP 3: Train Taichi Splatting ===")
# Ensure the folder structure is correct
if not os.path.exists("working_data/3d_scan"):
    print("❌ working_data/3d_scan not found. Check dataset.")
else:
    !python scripts/step3_train_splatting.py --project_path "working_data/3d_scan" --output_path "outputs/3d_scan/taichi_splatting"

In [ ]:
print("=== STEP 4: Export ===")
!python scripts/step4_export.py --input_parquet "outputs/3d_scan/taichi_splatting/model.parquet" --output_splat "outputs/3d_scan/taichi_splatting/model.splat"

In [ ]:
print("=== Compress Final Model for Download ===")
output_model_zip = "3d_splat_model.zip"

if os.path.exists("outputs/3d_scan/taichi_splatting"):
    !zip -r {output_model_zip} outputs/3d_scan/taichi_splatting
    
    from IPython.display import FileLink
    display(FileLink(output_model_zip))
else:
    print("❌ No output model found.")